<a href="https://colab.research.google.com/github/ayubsaid/comp1818-sentiment-analysis/blob/main/sentiment_coursework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COMP1818 Coursework: Symbolic vs Machine Learning vs Hybrid Sentiment Analysis of Reviews


In [ ]:
!pip install -q vaderSentiment datasets statsmodels transformers

In [ ]:
import os, time, warnings
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from datasets import load_dataset
from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split, cross_val_predict, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from statsmodels.stats.contingency_tables import mcnemar
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings("ignore", category=FutureWarning)
matplotlib.rcParams["pdf.fonttype"] = 42

SEED = 42
np.random.seed(SEED)
RUN_TRANSFORMER = True
TRANSFORMER_N = 500
os.makedirs("outputs", exist_ok=True)

In [ ]:
imdb = load_dataset("stanfordnlp/imdb")
train_df, test_df = pd.DataFrame(imdb["train"]), pd.DataFrame(imdb["test"])
X_train, y_train = train_df["text"], train_df["label"].values
X_test, y_test = test_df["text"], test_df["label"].values

yelp_full = load_dataset("fancyzhx/yelp_polarity", split="test").shuffle(seed=SEED)
yelp1, yelp2 = yelp_full.select(range(10000)), yelp_full.select(range(10000, 20000))
Y1_text, y_y1 = pd.Series(yelp1["text"]), np.array(yelp1["label"])
Y2_text, y_y2 = pd.Series(yelp2["text"]), np.array(yelp2["label"])

print("IMDb train/test:", X_train.shape[0], X_test.shape[0], np.bincount(y_train), np.bincount(y_test))
print("Yelp-1 / Yelp-2:", len(y_y1), len(y_y2), np.bincount(y_y1), np.bincount(y_y2))

In [ ]:
results = {}

def evaluate(name, y_true, y_pred, verbose=True):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    results[name] = {"Accuracy": acc, "Macro-F1": f1}
    if verbose:
        print(f"{name:34s} Accuracy={acc:.4f}  Macro-F1={f1:.4f}")

majority = pd.Series(y_train).mode()[0]
rng = np.random.default_rng(SEED)
evaluate("IMDb | Majority class", y_test, np.full(len(y_test), majority))
evaluate("IMDb | Random guess", y_test, rng.integers(0, 2, size=len(y_test)))
evaluate("Yelp-1 | Majority class", y_y1, np.full(len(y_y1), majority))
evaluate("Yelp-1 | Random guess", y_y1, rng.integers(0, 2, size=len(y_y1)))

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def vader_features(texts):
    return np.array([[s["neg"], s["neu"], s["pos"], s["compound"]]
                     for s in (analyzer.polarity_scores(t) for t in texts)])

def cached_vader(name, texts):
    path = f"outputs/vader_{name}.npy"
    if os.path.exists(path):
        return np.load(path)
    t0 = time.time()
    arr = vader_features(texts)
    np.save(path, arr)
    print(f"  scored {name}: {len(texts)} reviews in {time.time() - t0:.0f}s")
    return arr

V_train = cached_vader("imdb_train", X_train)
V_test = cached_vader("imdb_test", X_test)
V_y1 = cached_vader("yelp1", Y1_text)
V_y2 = cached_vader("yelp2", Y2_text)
print(V_train.shape, V_test.shape, V_y1.shape, V_y2.shape)

In [ ]:
evaluate("IMDb | VADER (threshold 0)", y_test, (V_test[:, 3] >= 0).astype(int))
print(confusion_matrix(y_test, (V_test[:, 3] >= 0).astype(int)))

In [ ]:
idx_tr, idx_val = train_test_split(np.arange(len(y_train)), test_size=0.2,
                                   stratify=y_train, random_state=SEED)
thresholds = np.arange(-0.5, 0.96, 0.05)
val_f1 = [f1_score(y_train[idx_val], (V_train[idx_val, 3] >= t).astype(int), average="macro")
          for t in thresholds]
best_t = float(thresholds[int(np.argmax(val_f1))])
print(f"Best threshold on validation data: {best_t:.2f}")

plt.figure(figsize=(5, 3.2))
plt.plot(thresholds, val_f1, marker="o")
plt.axvline(best_t, linestyle="--", color="gray")
plt.xlabel("Compound-score threshold")
plt.ylabel("Validation macro-F1")
plt.tight_layout()
plt.savefig("outputs/vader_threshold.pdf")
plt.show()

tuned_pred = (V_test[:, 3] >= best_t).astype(int)
evaluate("IMDb | VADER (tuned)", y_test, tuned_pred)
print(confusion_matrix(y_test, tuned_pred))

In [ ]:
tfidf0 = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
Xtr0 = tfidf0.fit_transform(X_train)
Xte0 = tfidf0.transform(X_test)
lr0 = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xtr0, y_train)
evaluate("IMDb | TF-IDF + LR (default)", y_test, lr0.predict(Xte0))

In [ ]:

grid = GridSearchCV(LogisticRegression(max_iter=1000, random_state=SEED),
                    {"C": [0.1, 1, 10, 100]}, cv=3, scoring="f1_macro", n_jobs=-1)
grid.fit(Xtr0, y_train)
print(pd.DataFrame(grid.cv_results_)[["param_C", "mean_test_score"]])
best_C = grid.best_params_["C"]
print("Best C:", best_C)


cv_scores = {}
for ng in [(1, 1), (1, 2)]:
    vec = TfidfVectorizer(max_features=50000, ngram_range=ng, sublinear_tf=True, min_df=2)
    A = vec.fit_transform(X_train)
    cv_scores[ng] = cross_val_score(LogisticRegression(C=best_C, max_iter=1000, random_state=SEED),
                                    A, y_train, cv=3, scoring="f1_macro").mean()
    print(ng, round(cv_scores[ng], 4))
best_ng = max(cv_scores, key=cv_scores.get)
print("Best n-gram range:", best_ng)


tfidf = TfidfVectorizer(max_features=50000, ngram_range=best_ng, sublinear_tf=True, min_df=2)
Xtr, Xte = tfidf.fit_transform(X_train), tfidf.transform(X_test)
lr_best = LogisticRegression(C=best_C, max_iter=1000, random_state=SEED).fit(Xtr, y_train)
lr_pred = lr_best.predict(Xte)
evaluate("IMDb | TF-IDF + LR (tuned)", y_test, lr_pred)

In [ ]:

Htr = hstack([Xtr, csr_matrix(V_train)]).tocsr()
Hte = hstack([Xte, csr_matrix(V_test)]).tocsr()
hyb = LogisticRegression(C=best_C, max_iter=1000, random_state=SEED).fit(Htr, y_train)
hyb_pred = hyb.predict(Hte)
evaluate("IMDb | Hybrid (feature-level)", y_test, hyb_pred)


base = LogisticRegression(C=best_C, max_iter=1000, random_state=SEED)
oof = cross_val_predict(base, Xtr, y_train, cv=5, method="predict_proba", n_jobs=-1)[:, 1]
S_train = np.column_stack([oof, V_train[:, 3]])
S_test = np.column_stack([lr_best.predict_proba(Xte)[:, 1], V_test[:, 3]])
meta = LogisticRegression(random_state=SEED).fit(S_train, y_train)
stack_pred = meta.predict(S_test)
print("Meta weights [LR prob, VADER compound]:", meta.coef_, meta.intercept_)
evaluate("IMDb | Hybrid (stacking)", y_test, stack_pred)

In [ ]:
def yelp_eval(tag, text, y, V):
    X = tfidf.transform(text)
    H = hstack([X, csr_matrix(V)]).tocsr()
    S = np.column_stack([lr_best.predict_proba(X)[:, 1], V[:, 3]])
    p = {"VADER (tuned)": (V[:, 3] >= best_t).astype(int),
         "TF-IDF + LR (tuned)": lr_best.predict(X),
         "Hybrid (feature-level)": hyb.predict(H),
         "Hybrid (stacking)": meta.predict(S)}
    for name, pred in p.items():
        evaluate(f"{tag} | {name}", y, pred)
    return p, X, H

y1_preds, Xy1, Hy1 = yelp_eval("Yelp-1", Y1_text, y_y1, V_y1)
print()
y2_preds, Xy2, Hy2 = yelp_eval("Yelp-2", Y2_text, y_y2, V_y2)
print("\nMean words/review  IMDb:", round(X_test.str.split().str.len().mean(), 1),
      " Yelp-1:", round(Y1_text.str.split().str.len().mean(), 1))

In [ ]:
def mcnemar_test(a, b, y):
    ca, cb = (a == y), (b == y)
    n_a_only, n_b_only = int(np.sum(ca & ~cb)), int(np.sum(~ca & cb))
    table = [[int(np.sum(ca & cb)), n_a_only], [n_b_only, int(np.sum(~ca & ~cb))]]
    return n_a_only, n_b_only, mcnemar(table, exact=True).pvalue

pairs = [("TF-IDF + LR (tuned)", "VADER (tuned)"),
         ("TF-IDF + LR (tuned)", "Hybrid (feature-level)"),
         ("TF-IDF + LR (tuned)", "Hybrid (stacking)"),
         ("Hybrid (stacking)", "Hybrid (feature-level)")]
imdb_preds = {"VADER (tuned)": tuned_pred, "TF-IDF + LR (tuned)": lr_pred,
              "Hybrid (feature-level)": hyb_pred, "Hybrid (stacking)": stack_pred}

mc_rows = []
for tag, P, y in [("IMDb", imdb_preds, y_test), ("Yelp-1", y1_preds, y_y1), ("Yelp-2", y2_preds, y_y2)]:
    for a, b in pairs:
        na, nb, p = mcnemar_test(P[a], P[b], y)
        mc_rows.append({"Test set": tag, "Model A": a, "Model B": b,
                        "Only A right": na, "Only B right": nb, "p-value": p})
mc = pd.DataFrame(mc_rows)
mc.to_csv("outputs/mcnemar_results.csv", index=False)
print(mc.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\n(p printed as 0 means p < 1e-300; report as p < 0.001)")

In [ ]:
sizes = [100, 500, 2000, 10000, 25000]
rows = []
for n in sizes:
    for s in [0, 1, 2]:
        if n == len(y_train) and s > 0:
            continue
        idx = np.random.default_rng(s).choice(len(y_train), size=n, replace=False)
        vec = TfidfVectorizer(max_features=50000, ngram_range=best_ng, sublinear_tf=True, min_df=2)
        A = vec.fit_transform(X_train.iloc[idx])
        B = vec.transform(X_test)
        m1 = LogisticRegression(C=best_C, max_iter=1000).fit(A, y_train[idx])
        f_lr = f1_score(y_test, m1.predict(B), average="macro")
        m2 = LogisticRegression(C=best_C, max_iter=1000).fit(
            hstack([A, csr_matrix(V_train[idx])]).tocsr(), y_train[idx])
        f_h = f1_score(y_test, m2.predict(hstack([B, csr_matrix(V_test)]).tocsr()), average="macro")
        rows.append({"n": n, "TF-IDF + LR": f_lr, "Hybrid (feature-level)": f_h})

lc = pd.DataFrame(rows).groupby("n").agg(["mean", "std"])
lc.to_csv("outputs/learning_curve_table.csv")
print(lc.round(4))

vader_f1 = results["IMDb | VADER (tuned)"]["Macro-F1"]
plt.figure(figsize=(5.5, 3.5))
for col in ["TF-IDF + LR", "Hybrid (feature-level)"]:
    plt.errorbar(sizes, lc[col]["mean"], yerr=lc[col]["std"].fillna(0), marker="o", capsize=3, label=col)
plt.axhline(vader_f1, linestyle="--", color="gray", label="VADER (tuned, no training)")
plt.xscale("log")
plt.xlabel("Number of labelled training reviews")
plt.ylabel("Test macro-F1")
plt.legend(loc="center right")
plt.tight_layout()
plt.savefig("outputs/learning_curve.pdf")
plt.show()

In [ ]:
rows = []
for w in [0.0, 0.1, 0.3, 1.0, 3.0]:
    Htr_w = hstack([Xtr, csr_matrix(V_train * w)]).tocsr()
    cv = cross_val_score(LogisticRegression(C=best_C, max_iter=1000, random_state=SEED),
                         Htr_w, y_train, cv=3, scoring="f1_macro").mean()
    m = LogisticRegression(C=best_C, max_iter=1000, random_state=SEED).fit(Htr_w, y_train)
    rows.append({"VADER weight": w, "CV F1 (IMDb train)": cv,
                 "IMDb test F1": f1_score(y_test, m.predict(hstack([Xte, csr_matrix(V_test * w)]).tocsr()), average="macro"),
                 "Yelp-1 test F1": f1_score(y_y1, m.predict(hstack([Xy1, csr_matrix(V_y1 * w)]).tocsr()), average="macro")})
wt = pd.DataFrame(rows)
wt.to_csv("outputs/weight_sweep_full.csv", index=False)
print(wt.round(4).to_string(index=False))

In [ ]:

rows = []
for n in [500, 2000]:
    for s in [0, 1, 2]:
        idx = np.random.default_rng(s).choice(len(y_train), size=n, replace=False)
        vec = TfidfVectorizer(max_features=50000, ngram_range=best_ng, sublinear_tf=True, min_df=2)
        A = vec.fit_transform(X_train.iloc[idx])
        B, Cy = vec.transform(X_test), vec.transform(Y1_text)
        for w in [0.0, 0.3, 1.0, 3.0]:
            m = LogisticRegression(C=best_C, max_iter=1000).fit(
                hstack([A, csr_matrix(V_train[idx] * w)]).tocsr(), y_train[idx])
            rows.append({"n": n, "weight": w, "seed": s,
                "IMDb F1": f1_score(y_test, m.predict(hstack([B, csr_matrix(V_test * w)]).tocsr()), average="macro"),
                "Yelp-1 F1": f1_score(y_y1, m.predict(hstack([Cy, csr_matrix(V_y1 * w)]).tocsr()), average="macro")})
ws = pd.DataFrame(rows)
small = ws.groupby(["n", "weight"])[["IMDb F1", "Yelp-1 F1"]].agg(["mean", "std"])
small.to_csv("outputs/weight_sweep_small_n.csv")
print(small.round(4))

In [ ]:
lengths = X_test.str.split().str.len().values
q = pd.qcut(lengths, 4)

acc_len = pd.DataFrame({"length quartile": q,
                        "VADER (tuned)": tuned_pred == y_test,
                        "TF-IDF + LR": lr_pred == y_test,
                        "Hybrid": hyb_pred == y_test}).groupby("length quartile", observed=True).mean()
print(acc_len.round(4))
acc_len.to_csv("outputs/accuracy_by_length.csv")

comp = np.abs(V_test[:, 3])
sat = pd.DataFrame({"length quartile": q, "mean |compound|": comp,
                    "share |compound|>0.9": comp > 0.9}).groupby("length quartile", observed=True).mean()
print(sat.round(3))
sat.to_csv("outputs/vader_saturation_by_length.csv")

plt.figure(figsize=(5.2, 3.3))
xs = np.arange(len(acc_len))
for col, mk in zip(acc_len.columns, ["o", "s", "^"]):
    plt.plot(xs, acc_len[col].values, marker=mk, label=col)
plt.xticks(xs, ["Q1\n(shortest)", "Q2", "Q3", "Q4\n(longest)"])
plt.xlabel("Review-length quartile (IMDb test set)")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/accuracy_by_length.pdf")
plt.show()

In [ ]:
rs = np.random.default_rng(SEED)
def show(mask, title, k=5):
    print(f"\n== {title}: {int(mask.sum())} reviews ==")
    for i in rs.choice(np.where(mask)[0], size=k, replace=False):
        print(f"[true={y_test[i]} vader={V_test[i, 3]:+.2f} words={lengths[i]}]",
              repr(X_test.iloc[i][:300]), "\n")

show((tuned_pred != y_test) & (lr_pred == y_test), "VADER wrong, LR right")
show((tuned_pred == y_test) & (lr_pred != y_test), "VADER right, LR wrong")

In [ ]:
if RUN_TRANSFORMER:
    from transformers import pipeline
    import torch
    clf = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english",
                   device=0 if torch.cuda.is_available() else -1)

    def bert_predict(texts, bs=16, chunk=100):
        texts, preds, t0 = list(texts), [], time.time()
        for i in range(0, len(texts), chunk):
            out = clf(texts[i:i + chunk], batch_size=bs, truncation=True, max_length=256)
            preds += [1 if o["label"] == "POSITIVE" else 0 for o in out]
            print(f"{len(preds)}/{len(texts)} done, {time.time() - t0:.0f}s elapsed")
        return np.array(preds)

    N = TRANSFORMER_N
    ri = np.random.default_rng(SEED).choice(len(y_test), N, replace=False)
    ry = np.random.default_rng(SEED).choice(len(y_y1), N, replace=False)
    bi, by = bert_predict(X_test.iloc[ri]), bert_predict(Y1_text.iloc[ry])

    evaluate("Sub IMDb | DistilBERT (SST-2)", y_test[ri], bi)
    evaluate("Sub IMDb | VADER (tuned)", y_test[ri], tuned_pred[ri])
    evaluate("Sub IMDb | TF-IDF + LR", y_test[ri], lr_pred[ri])
    evaluate("Sub IMDb | Hybrid", y_test[ri], hyb_pred[ri])
    evaluate("Sub Yelp-1 | DistilBERT (SST-2)", y_y1[ry], by)
    evaluate("Sub Yelp-1 | VADER (tuned)", y_y1[ry], y1_preds["VADER (tuned)"][ry])
    evaluate("Sub Yelp-1 | TF-IDF + LR", y_y1[ry], y1_preds["TF-IDF + LR (tuned)"][ry])
    evaluate("Sub Yelp-1 | Hybrid", y_y1[ry], y1_preds["Hybrid (feature-level)"][ry])
else:
    print("Transformer reference skipped (RUN_TRANSFORMER = False).")

## 13. Final tables and figures for the report

In [ ]:
res = pd.DataFrame(results).T.round(4)
res.to_csv("outputs/all_results.csv")
print(res.to_string())


names = ["VADER (tuned)", "TF-IDF + LR (tuned)", "Hybrid (feature-level)", "Hybrid (stacking)"]
imdb_acc = [results["IMDb | " + n]["Accuracy"] for n in names]
yelp_acc = [results["Yelp-1 | " + n]["Accuracy"] for n in names]
x = np.arange(len(names)); wdt = 0.38
plt.figure(figsize=(5.6, 3.4))
plt.bar(x - wdt / 2, imdb_acc, wdt, label="IMDb (in-domain)")
plt.bar(x + wdt / 2, yelp_acc, wdt, label="Yelp-1 (cross-domain)")
plt.xticks(x, ["VADER\n(tuned)", "TF-IDF\n+ LR", "Hybrid\n(feature)", "Hybrid\n(stacking)"])
plt.ylabel("Accuracy"); plt.ylim(0.5, 1.0)
plt.legend(loc="upper left")
plt.tight_layout()
plt.savefig("outputs/main_results.pdf")
plt.show()

print("\nFiles in outputs/:", sorted(os.listdir("outputs")))